In [ ]:
!pip install bs4 requests lxml pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

current_page = 1
max_pages = 450
data = []
previous_count = 0

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}


options = Options()
options.headless = True 
options.add_argument(f"user-agent={headers['User-Agent']}") 
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)

while current_page <= max_pages:
    print(f"\nCurrently scraping page: {current_page}")
    url = f"https://www.property24.co.ke/property-for-sale-in-nairobi-p95?Page={current_page}"
    page = requests.get(url, headers=headers)
    if page.status_code != 200:
        print("Stopping: Page not found or blocked.")
        break
    soup = BeautifulSoup(page.text, "html.parser")
    all_listings = soup.find_all("div", class_="col-xs-9")
    print(f"Found {len(all_listings)} listings on page {current_page}")
    if not all_listings:
        print("No listings found. Stopping.")
        break
    for listing in all_listings:
        
        item = {}
        title_tag = listing.find("span", class_="p24_propertyTitle")
        if title_tag:
            item["Title"] = title_tag.text.strip()
           
            link_tag = title_tag.find_parent("a")
            if link_tag and "href" in link_tag.attrs:
                detail_url = "https://www.property24.co.ke" + link_tag["href"]
                item["Detail URL"] = detail_url 
            else:
                print(f"Warning: No detail link found for title '{item['Title']}'")
                continue 
        location_tag = listing.find("span", class_="p24_location")
        item["Location"] = location_tag.text.strip() if location_tag else None
        bedrooms_tag = listing.find("span", title="Bedrooms")
        item["Bedrooms"] = bedrooms_tag.text.strip() if bedrooms_tag else None
        bathrooms_tag = listing.find("span", title="Bathrooms")
        item["Bathrooms"] = bathrooms_tag.text.strip() if bathrooms_tag else None
        size_tag = listing.find("span", class_="p24_size")
        item["House Size"] = size_tag.text.strip() if size_tag else None
        price_tag = listing.find("span", class_="p24_price")
        item["Price"] = price_tag.text.strip() if price_tag else None
       
        if "Detail URL" in item:
            try:
                driver.get(item["Detail URL"])
                time.sleep(3) 
                detail_soup = BeautifulSoup(driver.page_source, "html.parser")
                
                type_key = detail_soup.find(lambda tag: tag.name == "div" and tag.string and "Type of Property" == tag.string.strip())
                if type_key:
                    value_tag = type_key.find_next_sibling("div")
                    if value_tag:
                        item["Property Type"] = value_tag.text.strip()
                    else:
                        print(f"Warning: No value found for 'Type of Property' in '{item['Title']}'")
                else:
                    print(f"Warning: 'Type of Property' key not found in '{item['Title']}'")
                
                date_key = detail_soup.find(lambda tag: tag.name == "div" and tag.string and tag.string.strip() in ["List Date", "Listing Date"])
                if date_key:
                    value_tag = date_key.find_next_sibling("div")
                    if value_tag:
                        item["Listing Date"] = value_tag.text.strip()
                    else:
                        print(f"Warning: No value found for 'List Date' in '{item['Title']}'")
                else:
                    print(f"Warning: 'List Date' key not found in '{item['Title']}'")

                # ───────────────────────────────────────────────────────────────
                # AMENITIES EXTRACTION FROM id="readMoreDescription"
                amenities = []

                desc_container = detail_soup.find(id="readMoreDescription")
                
                if desc_container:
                    full_text = desc_container.get_text(separator="\n", strip=True)
                    lines = [line.strip() for line in full_text.split("\n") if line.strip()]

                    in_amenities = False
                    for line in lines:
                        lower_line = line.lower()

                        
                        if "amenities" in lower_line:
                            in_amenities = True
                            continue

                        
                        if in_amenities and any(
                            section in lower_line for section in [
                                "payment plan", "project completion", "note:", "tenure:", "construction"
                            ]
                        ):
                            break

                       
                        if in_amenities:
                            
                            if (len(line) < 8 or 
                                line.startswith("*") or 
                                "note:" in lower_line or 
                                "option" in lower_line or 
                                "deposit" in lower_line or 
                                "leasehold" in lower_line or 
                                "kes" in lower_line.lower()):
                                continue

                            cleaned = line.replace(">", "").strip()
                            if cleaned and len(cleaned) > 6:
                                amenities.append(cleaned)

                
                if not amenities:
                    full_text = detail_soup.get_text(separator="\n", strip=True)
                    lines = [line.strip() for line in full_text.split("\n") if line.strip()]
                    for line in lines:
                        if line.startswith(">") and len(line) > 10:
                            feature = line[1:].strip()
                            amenities.append(feature)

             
                exclude_keywords = [
                    "photo grid", "video", "property overview", "listing number", "type of property",
                    "street address", "list date", "erf size", "floor area", "price per m²",
                    "service charge", "pets allowed", "external features", "garage", "parking",
                    "garden", "video tour", "show contact", "view all", "send listing",
                    "cancel", "print", "share", "nairobi", "westlands", "kilimani", "home"
                ]

                clean_amenities = []
                for a in amenities:
                    a_clean = a.strip()
                    lower_a = a_clean.lower()
                    if (a_clean and 
                        len(a_clean) > 8 and 
                        not any(kw in lower_a for kw in exclude_keywords)):
                        clean_amenities.append(a_clean)

                clean_amenities = list(dict.fromkeys(clean_amenities))

                if clean_amenities:
                    item["Amenities"] = ", ".join(clean_amenities)
                else:
                    item["Amenities"] = None
                # ───────────────────────────────────────────────────────────────

            except Exception as e:
                print(f"Exception fetching detail for '{item['Title']}': {e}")
           
        time.sleep(10 + random.uniform(0, 5)) 
        
        data.append(item)
    print(f"Total records collected so far: {len(data)}")
    if len(data) == previous_count:
        print("No new data added. Stopping scraper.")
        break
    previous_count = len(data)
    current_page += 1
    time.sleep(2)

driver.quit()

df = pd.DataFrame(data)
df.dropna(how="all", inplace=True)
df.to_csv("kenya_property24_sales.csv", index=False)
print("\nScraping complete. Data saved to 'kenya_property24_sales.csv'")
print(f"Final total records: {len(df)}")

In [ ]:
data_dictionary = pd.DataFrame([
    {
        "Column Name": "Title",
        "Description": "Full title of the property listing",
        "Data Type": "String",
        "Source": "Listing page"
    },
    {
        "Column Name": "Detail URL",
        "Description": "URL to the full property detail page",
        "Data Type": "String (URL)",
        "Source": "Listing page"
    },
    {
        "Column Name": "Location",
        "Description": "Suburb or area where property is located",
        "Data Type": "String",
        "Source": "Listing page"
    },
    {
        "Column Name": "Bedrooms",
        "Description": "Number of bedrooms",
        "Data Type": "Numeric",
        "Source": "Listing page"
    },
    {
        "Column Name": "Bathrooms",
        "Description": "Number of bathrooms",
        "Data Type": "Numeric",
        "Source": "Listing page"
    },
    {
        "Column Name": "House Size",
        "Description": "Size of the property (usually square meters)",
        "Data Type": "String",
        "Source": "Listing page"
    },
    {
        "Column Name": "Price",
        "Description": "Listed sale price of the property",
        "Data Type": "String (KES)",
        "Source": "Listing page"
    },
    {
        "Column Name": "Property Type",
        "Description": "Type of property (Apartment, House, Villa, etc.)",
        "Data Type": "String",
        "Source": "Detail page"
    },
    {
        "Column Name": "Listing Date",
        "Description": "Date the property was listed",
        "Data Type": "Date (String format)",
        "Source": "Detail page"
    },
    {
        "Column Name": "Amenities",
        "Description": "List of amenities/features extracted from description",
        "Data Type": "String (Comma-separated)",
        "Source": "Detail page description"
    }
])

data_dictionary.to_csv("kenya_property24_data_dictionary.csv", index=False)

print("Data dictionary saved as 'kenya_property24_data_dictionary.csv'")